# ADMET → AGP/CD → QSVT → Triple Alpha
This notebook curates ADMET data, constructs one molecule-specific counterdiabatic Hamiltonian, prepares an ordered runtime input stream, and submits it to an existing Triple Alpha deployment.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json, os, sys, time, requests
ROOT = Path.cwd().resolve()
if not (ROOT/'src'/'admet_submission_entry.py').is_file(): raise FileNotFoundError('Run from HORIZON_PROJECTS_FINAL_SUBMISSION')
sys.path.insert(0, str(ROOT/'src'))
import admet_submission_entry as entry
from classical_preprocessing import AGPSettings, recalculate_counterdiabatic_hamiltonian
from full_curation_from_notebook import run_complete_curation, molecule_problem_input


## Configuration
Set the dataset, physical parameters, execution settings, and API configuration below.

In [ ]:
DATASET='BBB_Martins'; SAMPLE_INDEX=0
INTERPOLATION=0.5; LAMBDA_DOT=1.0
PHYSICAL_TIME=0.1; EPSILON=0.001; NUM_SHOTS=10000
POLL_SECONDS=8; TIMEOUT_SECONDS=1800
API_KEY = os.environ.get('TRIPLE_ALPHA_API_KEY', '')
RUN_URL = os.environ.get('TRIPLE_ALPHA_RUN_URL', '')
if not API_KEY or not RUN_URL: raise RuntimeError('Configure the Triple Alpha API credentials')


## Classical preprocessing and AGP/CD recalculation

In [ ]:
CURATION_NOTEBOOK=ROOT/'pipeline'/'ADMET_future_curation_complete.ipynb'
if not CURATION_NOTEBOOK.is_file(): raise FileNotFoundError(CURATION_NOTEBOOK)
curation=run_complete_curation(CURATION_NOTEBOOK)
curated_adapter,feature_vector,curation_provenance=molecule_problem_input(curation,DATASET,SAMPLE_INDEX)
payload=recalculate_counterdiabatic_hamiltonian(curated_adapter, DATASET, feature_vector, AGPSettings(interpolation=INTERPOLATION, lambda_dot=LAMBDA_DOT, interaction_term_limit=None))
payload['sample_index']=SAMPLE_INDEX
payload['curation_provenance']=curation_provenance
print('Complete curation:',curation_provenance)
print('Counterdiabatic terms:',len(payload['pauli_terms']),payload['agp_metadata'])


## Validate and prepare runtime inputs

In [ ]:
circuit_dir,qsvt=entry.prepare_runtime_inputs(payload, entry.DEFAULT_OUTPUT_DIR, PHYSICAL_TIME, EPSILON)
validator=entry.build_validator(payload)
runtime_doc=qsvt['synchronization']['runtime_document']
runtime_inputs=runtime_doc['inputs']
assert qsvt['synchronization']['status']=='synchronized'
assert runtime_doc['input_count']==len(runtime_inputs)
print({'terms':validator.num_terms,'system_qubits':validator.num_system_qubits,'index_qubits':validator.num_index_qubits,'lambda':qsvt['lambda_value'],'qsvt_degree':qsvt['polynomial_degree'],'phases':len(qsvt['phase_angles']),'inputs':len(runtime_inputs),'sha256':runtime_doc['inputs_sha256']})


## Triple Alpha API client

In [ ]:
def endpoint(path):
    base=RUN_URL.rstrip('/'); suffix=path.removeprefix('run/')
    return f'{base}/{suffix}' if base.endswith('/run') else f'{base}/run/{suffix}'
def post(path, body):
    method='PATCH' if path=='run/update_input' else 'POST'
    response=requests.request(method, endpoint(path), headers={'X-Api-Key':API_KEY,'Content-Type':'application/json'}, json=body, timeout=60)
    if not response.ok: raise RuntimeError(f'{method} {response.url} returned {response.status_code}: {response.text}')
    return response.json()
def wait_for(job_id, accepted):
    deadline=time.monotonic()+TIMEOUT_SECONDS; previous=None
    while time.monotonic()<deadline:
        data=post('run/get_status', {'job_id':job_id}); status=data.get('job_status')
        if status!=previous: print('status:',status); previous=status
        if status in accepted or status in {'successful','failed','cancelled','canceled'}: return data
        time.sleep(POLL_SECONDS)
    raise TimeoutError(job_id)


## Create run, supply inputs, execute, and retrieve results

In [ ]:
created=post('run/create', {'num_shots':NUM_SHOTS})
job_id=created.get('job_id')
if not job_id: raise RuntimeError(f'No job_id in create response: {created}')
print('job_id:',job_id)
ready=wait_for(job_id, {'awaiting_input'})
if ready.get('job_status')!='awaiting_input': raise RuntimeError(ready)
print(post('run/update_input', {'job_id':job_id,'inputs':runtime_inputs}))
finished=wait_for(job_id, {'successful','failed'})
if finished.get('job_status')!='successful': raise RuntimeError(finished)
result_data=post('run/get_results', {'job_id':job_id})
print('completed:',job_id,result_data.get('job_status'))
result_data
